# Mechanistic Interpretability Pipeline - RunPod

This notebook runs the complete mechanistic interpretability pipeline on RunPod.

**Pipeline stages:**
1. **Random Baseline Analysis**: Runs DLA, Patching, Gradient on random positions as a control.
   - Uses random token positions defined in config (default: [8, 16])
   - Filters for temperature 0.6 runs only
2. **Token Impact Identification**: Finds branching points where token choice matters most
3. **Direct Logit Attribution (DLA)**: Layer-wise attribution of logit differences
4. **Activation Patching**: Causal intervention via activation replacement
5. **Gradient Attribution**: Sensitivity analysis via gradient computation

**GPU-optimized**: Uses float16 mixed precision and efficient memory management

## Step 1: Clone Repository and Install Dependencies

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Clone repository
repo_url = "https://github.com/dude123studios/Sampling1.git"
repo_dir = Path("/workspace/Sampling1")

if not repo_dir.exists():
    print(f"Cloning repository from {repo_url}...")
    subprocess.run(["git", "clone", "--depth", "1", repo_url, str(repo_dir)], check=True)
    print(f"✓ Repository cloned to {repo_dir}")
else:
    print(f"Repository already exists at {repo_dir}")

# Change to repo directory
os.chdir(repo_dir)
print(f"Working directory: {os.getcwd()}")

In [ ]:
!pip install tqdm transformers accelerate

In [ ]:
# Verify GPU availability
import torch

print("\n" + "="*60)
print("GPU INFORMATION")
print("="*60)
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"Number of GPUs: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"\nGPU {i}: {props.name}")
        print(f"  Total memory: {props.total_memory / 1e9:.2f} GB")
        print(f"  CUDA cores: {props.multi_processor_count * 128}")

## Step 2: Verify Data and Configurations

In [ ]:
# Check if sweep data exists
sweep_dir = repo_dir / "results/sweeps"
print("\n" + "="*60)
print("DATA VERIFICATION")
print("="*60)

print(f"\nLooking for sweep data in: {sweep_dir}")
if sweep_dir.exists():
    subdirs = list(sweep_dir.glob("*/"))
    print(f"✓ Found {len(subdirs)} sweep directories")
    
    # List sweep directories
    for subdir in subdirs[:5]:  # Show first 5
        log_file = subdir / "log.jsonl"
        if log_file.exists():
            with open(log_file) as f:
                lines = f.readlines()
            print(f"  - {subdir.name}: {len(lines)} entries")
else:
    print(f"✗ Sweep directory not found: {sweep_dir}")
    print("  You need to run the sweep experiments first or upload sweep data")

In [ ]:
# Check configurations
import yaml

config_dir = repo_dir / "configs/mech_interp"
print(f"\nConfiguration directory: {config_dir}")

if config_dir.exists():
    configs = list(config_dir.glob("*.yaml"))
    print(f"✓ Found {len(configs)} configuration files:")
    for config_file in configs:
        print(f"  - {config_file.name}")
        with open(config_file) as f:
            cfg = yaml.safe_load(f)
        if 'name' in cfg:
            print(f"    ({cfg['name']})")
else:
    print(f"✗ Configuration directory not found: {config_dir}")

## Step 3a: Random Baseline - PATCHING ONLY (Qwen 3 8B)

In [ ]:
# Run ONLY PATCHING for Qwen Random Baseline
print("\n" + "="*60)
print("RANDOM BASELINE: PATCHING-ONLY (QWEN)")
print("="*60)

script_path = repo_dir / "mech_interp/random_baseline_analysis.py"
config_path = repo_dir / "configs/mech_interp/dla_patching_config.yaml"

if not script_path.exists():
    print(f"✗ Script not found: {script_path}")
else:
    print(f"Running script: {script_path}")
    
    # Run only patching mode
    !python {script_path} --config {config_path} --modes patching
    
    # Rename output
    src = repo_dir / "mech_interp/random_baseline_results/random_baseline_full.json"
    dst = repo_dir / "mech_interp/random_baseline_results/random_baseline_qwen_patching.json"
    if src.exists():
        src.rename(dst)
        print(f"Renamed results to {dst.name}")
    
    print("Done.")

## Step 3b: Random Baseline - GRADIENT ONLY (Qwen 3 8B)

In [ ]:
# Run ONLY GRADIENT for Qwen Random Baseline
print("\n" + "="*60)
print("RANDOM BASELINE: GRADIENT-ONLY (QWEN)")
print("="*60)

script_path = repo_dir / "mech_interp/random_baseline_analysis.py"
config_path = repo_dir / "configs/mech_interp/dla_patching_config.yaml"

if not script_path.exists():
    print(f"✗ Script not found: {script_path}")
else:
    print(f"Running script: {script_path}")
    
    # Run only gradient mode
    !python {script_path} --config {config_path} --modes gradient
    
    # Rename output
    src = repo_dir / "mech_interp/random_baseline_results/random_baseline_full.json"
    dst = repo_dir / "mech_interp/random_baseline_results/random_baseline_qwen_gradient.json"
    if src.exists():
        src.rename(dst)
        print(f"Renamed results to {dst.name}")
    
    print("Done.")

## Step 3c: Random Baseline - PATCHING ONLY (DeepSeek)

In [ ]:
# Run ONLY PATCHING for DeepSeek Random Baseline
print("\n" + "="*60)
print("RANDOM BASELINE: PATCHING-ONLY (DEEPSEEK)")
print("="*60)

script_path = repo_dir / "mech_interp/random_baseline_analysis.py"
config_path = repo_dir / "configs/mech_interp/dla_patching_config_deepseek.yaml"

if not script_path.exists():
    print(f"✗ Script not found")
else:
    print(f"Running script: {script_path}")
    
    # Run only patching mode
    !python {script_path} --config {config_path} --modes patching
    
    # Rename output
    src = repo_dir / "mech_interp/random_baseline_results/random_baseline_full.json"
    dst = repo_dir / "mech_interp/random_baseline_results/random_baseline_deepseek_patching.json"
    if src.exists():
        src.rename(dst)
        print(f"Renamed results to {dst.name}")
    
    print("Done.")

## Step 3d: Random Baseline - GRADIENT ONLY (DeepSeek)

In [ ]:
# Run ONLY GRADIENT for DeepSeek Random Baseline
print("\n" + "="*60)
print("RANDOM BASELINE: GRADIENT-ONLY (DEEPSEEK)")
print("="*60)

script_path = repo_dir / "mech_interp/random_baseline_analysis.py"
config_path = repo_dir / "configs/mech_interp/dla_patching_config_deepseek.yaml"

if not script_path.exists():
    print(f"✗ Script not found")
else:
    print(f"Running script: {script_path}")
    
    # Run only gradient mode
    !python {script_path} --config {config_path} --modes gradient
    
    # Rename output
    src = repo_dir / "mech_interp/random_baseline_results/random_baseline_full.json"
    dst = repo_dir / "mech_interp/random_baseline_results/random_baseline_deepseek_gradient.json"
    if src.exists():
        src.rename(dst)
        print(f"Renamed results to {dst.name}")
    
    print("Done.")

## Step 3e: Random Baseline - DLA ONLY (DeepSeek)

In [ ]:
# Run ONLY DLA for DeepSeek Random Baseline
print("\n" + "="*60)
print("RANDOM BASELINE: DLA-ONLY (DEEPSEEK)")
print("="*60)

script_path = repo_dir / "mech_interp/random_baseline_analysis.py"
config_path = repo_dir / "configs/mech_interp/dla_patching_config_deepseek.yaml"

if not script_path.exists():
    print(f"✗ Script not found")
else:
    print(f"Running script: {script_path}")
    
    # Run only dla mode
    !python {script_path} --config {config_path} --modes dla
    
    # Rename output
    src = repo_dir / "mech_interp/random_baseline_results/random_baseline_full.json"
    dst = repo_dir / "mech_interp/random_baseline_results/random_baseline_deepseek_dla.json"
    if src.exists():
        src.rename(dst)
        print(f"Renamed results to {dst.name}")
    
    print("Done.")

## Step 4: Run Token Impact Identification

In [ ]:
print("\n" + "="*60)
print("STAGE 1: TOKEN IMPACT IDENTIFICATION")
print("="*60)

script_path = repo_dir / "mech_interp/token_impact.py"
config_path = repo_dir / "configs/mech_interp/token_impact_config.yaml"

if not script_path.exists():
    print(f"✗ Script not found: {script_path}")
else:
    # Using !python for direct execution
    !python {script_path} --config {config_path}
    
    # Check output
    results_dir = repo_dir / "mech_interp/token_impact_results"
    if results_dir.exists():
        results_files = list(results_dir.glob("*.json"))
        print(f"✓ Generated {len(results_files)} output files:")
        for f in results_files:
            size_kb = f.stat().st_size / 1024
            print(f"    - {f.name} ({size_kb:.1f} KB)")

## Step 5: Run DLA (Direct Logit Attribution)

In [ ]:
print("\n" + "="*60)
print("STAGE 2: DIRECT LOGIT ATTRIBUTION (DLA)")
print("="*60)

token_impact_file = repo_dir / "mech_interp/token_impact_results/token_impact_results.json"
script_path = repo_dir / "mech_interp/dla.py"
config_path = repo_dir / "configs/mech_interp/dla_patching_config.yaml"

if not token_impact_file.exists():
    print(f"✗ Token impact results not found: {token_impact_file}")
    print("Run Stage 1 first.")
else:
    print(f"✓ Found token impact results")
    
    # Run DLA using !python
    !python {script_path} --config {config_path} --branching-points {token_impact_file}
    
    # Check output
    dla_dir = repo_dir / "mech_interp/dla_results"
    if dla_dir.exists():
        results_files = list(dla_dir.glob("*.json"))
        print(f"✓ Generated {len(results_files)} output files")

## Step 6: Run Activation Patching

In [ ]:
print("\n" + "="*60)
print("STAGE 3: ACTIVATION PATCHING")
print("="*60)

script_path = repo_dir / "mech_interp/patching.py"
config_path = repo_dir / "configs/mech_interp/dla_patching_config.yaml"

if not token_impact_file.exists():
    print(f"✗ Token impact results not found")
else:
    # Run patching using !python
    !python {script_path} --config {config_path} --branching-points {token_impact_file}
    
    # Check output
    patching_dir = repo_dir / "mech_interp/patching_results"
    if patching_dir.exists():
        results_files = list(patching_dir.glob("*.json"))
        print(f"✓ Generated {len(results_files)} output files")

## Step 7: Run Gradient Attribution

In [ ]:
print("\n" + "="*60)
print("STAGE 4: GRADIENT ATTRIBUTION")
print("="*60)

script_path = repo_dir / "mech_interp/gradient.py"
config_path = repo_dir / "configs/mech_interp/dla_patching_config.yaml"

if not token_impact_file.exists():
    print(f"✗ Token impact results not found")
else:
    # Run gradient using !python
    !python {script_path} --config {config_path} --branching-points {token_impact_file}
    
    # Check output
    gradient_dir = repo_dir / "mech_interp/gradient_results"
    if gradient_dir.exists():
        results_files = list(gradient_dir.glob("*.json"))
        print(f"✓ Generated {len(results_files)} output files")

In [ ]:
# Trajectory Bifurcation Analysis - Uses existing sweep data
print("\n" + "="*60)
print("TRAJECTORY BIFURCATION ANALYSIS")
print("="*60)

# Run for Qwen
print("\n--- Qwen3-8B ---")
qwen_config = repo_dir / "configs/mech_interp/bifurcation_config.yaml"
qwen_script = repo_dir / "mech_interp/bifurcation_analysis.py"

if qwen_script.exists() and qwen_config.exists():
    !python {qwen_script} {qwen_config}
else:
    print("✗ Bifurcation script or config not found")

# Run for DeepSeek
print("\n--- DeepSeek-Qwen3-8B ---")
deepseek_config = repo_dir / "configs/mech_interp/bifurcation_deepseek_config.yaml"

if qwen_script.exists() and deepseek_config.exists():
    !python {qwen_script} {deepseek_config}
else:
    print("✗ DeepSeek config not found")

# Check outputs
bifurcation_dir = repo_dir / "mech_interp/bifurcation_results"
if bifurcation_dir.exists():
    results = list(bifurcation_dir.glob("*.png"))
    print(f"\n✓ Generated {len(results)} bifurcation plots:")
    for f in results:
        print(f"    - {f.name}")

## Step 8: Trajectory Bifurcation Analysis

## Step 8: Summary and Results

In [ ]:
print("\n" + "="*60)
print("PIPELINE SUMMARY")
print("="*60)

results_dirs = [
    ("Random Baseline", repo_dir / "mech_interp/random_baseline_results"),
    ("Token Impact", repo_dir / "mech_interp/token_impact_results"),
    ("DLA", repo_dir / "mech_interp/dla_results"),
    ("Patching", repo_dir / "mech_interp/patching_results"),
    ("Gradient", repo_dir / "mech_interp/gradient_results"),
    ("Bifurcation", repo_dir / "mech_interp/bifurcation_results"),
]

print("\nOutput files generated:")
for stage_name, stage_dir in results_dirs:
    if stage_dir.exists():
        files = list(stage_dir.glob("*.json")) + list(stage_dir.glob("*.png"))
        print(f"\n✓ {stage_name}:")
        for f in files:
            size_kb = f.stat().st_size / 1024
            print(f"    - {f.name} ({size_kb:.1f} KB)")
    else:
        print(f"\n✗ {stage_name}: No results directory")

## Step 9: Download Results (if using RunPod)

In [ ]:
# Create a compressed archive of results
import zipfile
import json
from datetime import datetime

print("\n" + "="*60)
print("PREPARING RESULTS FOR DOWNLOAD")
print("="*60)

results_archive = repo_dir / f"mech_interp_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.zip"

print(f"\nCreating archive: {results_archive.name}")

with zipfile.ZipFile(results_archive, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for stage_name, stage_dir in results_dirs:
        if stage_dir.exists():
            for result_file in list(stage_dir.glob("*.json")) + list(stage_dir.glob("*.png")):
                arcname = f"mech_interp_results/{stage_name.lower()}/{result_file.name}"
                zipf.write(result_file, arcname=arcname)
                print(f"  Added: {arcname}")

if results_archive.exists():
    size_mb = results_archive.stat().st_size / (1024 * 1024)
    print(f"\n✓ Archive created: {results_archive.name} ({size_mb:.2f} MB)")
    print(f"\nTo download on RunPod, use the file browser or:")
    print(f"  sftp or download {results_archive.name}")
else:
    print("✗ Failed to create archive")

In [ ]:
# Print final summary
print("\n" + "="*60)
print("EXECUTION COMPLETE")
print("="*60)
print("\nResults locations:")
for stage_name, stage_dir in results_dirs:
    if stage_dir.exists():
        print(f"  ✓ {stage_name}: {stage_dir}")
    else:
        print(f"  - {stage_name}: Not generated")

print("\nNext steps:")
print("  1. Download the results archive")
print("  2. Analyze the JSON files for insights")
print("  3. Visualize layer-wise contributions and sensitivity")
print("\nDocumentation: See README.md in the repository")